# Baseline Experiment
- **Group Name**: MVPs
- **Course:** Advanced Deep Learning - AIGC-5500-0NA
- **Professor**: Hossein Pourmodheji
- **Owner:** Nouroz Ali
---

Starter notebook: shared data loading and batch verification are provided; model implementation, training, and results remain to be completed. No results are fabricated.

Reuse the prepared Enron partitions and training vocabulary. Select configurations on validation data and reserve test evaluation for frozen final models.

## 1. Initial Configuration

### 1.1. Imports, Paths, and Device

In [57]:
# Import required libraries
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from torch import nn

# Locate the repository from either its root or the notebooks folder
current_dir = Path.cwd().resolve()
PROJECT_ROOT = current_dir if (current_dir / 'src').is_dir() else current_dir.parent
if not (PROJECT_ROOT / 'src').is_dir():
    raise FileNotFoundError('Place this notebook in the repository notebooks folder.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Import shared project modules
from src.config import BATCH_SIZE, TRAINING_SEEDS
from src.data import load_prepared_data
from src.reproducibility import set_seed
from src.text import EncodedTextDataset, make_data_loader

# Use CUDA when it is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Print the runtime and selected device
print('Python:', sys.version.split()[0])
print('Interpreter:', sys.executable)
print('PyTorch:', torch.__version__)
print('Device:', device)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

Python: 3.12.14
Interpreter: c:\anaconda3\envs\ml-env\python.exe
PyTorch: 2.14.0+cpu
Device: cpu


### 1.2. Experiment Configuration

In [58]:
# Name this experiment and its owner
MODEL_NAME = 'baseline'
OWNER = 'Nouroz Ali'

# Reset randomness before the first run
set_seed(TRAINING_SEEDS[0])

# Point to the shared prepared data and this model's output folders
PREPARED_DATA_DIR = PROJECT_ROOT / 'data' / 'processed' / 'enron'
RESULTS_DIR = PROJECT_ROOT / 'results' / 'enron' / MODEL_NAME
CHECKPOINT_DIR = PROJECT_ROOT / 'checkpoints' / 'enron' / MODEL_NAME

# Show the shared training seeds and batch size
print('Training seeds:', TRAINING_SEEDS)
print('Batch size:', BATCH_SIZE)

Training seeds: (42, 123, 2026)
Batch size: 64


## 2. Load Shared Prepared Data

Run `01_data_preparation.ipynb` once before this notebook. The saved preprocessing metadata controls sequence length; this notebook does not refit the vocabulary.

In [59]:
# Load the saved splits, vocabulary, and metadata
prepared = load_prepared_data(PREPARED_DATA_DIR)

# Keep the cleaned partitions
splits = prepared['splits']

# Keep the training vocabulary
vocabulary = prepared['vocabulary']

# Keep the preprocessing configuration
metadata = prepared['metadata']

# Use the saved sequence length instead of choosing a new one
MAX_LENGTH = metadata['max_length']

# Confirm each split still has the saved number of rows
for name, expected_size in metadata['split_sizes'].items():
    assert len(splits[name]) == expected_size

# Show the loaded split sizes
pd.DataFrame([{'split': name, 'rows': len(part)} for name, part in splits.items()])



,split,rows
0,train,22524
1,validation,5632
2,test,1981


In [60]:

print("Training data type:", type(splits["train"]))
print("Training column names:", splits["train"].column_names)

print("\nSample training records:")
display(pd.DataFrame(splits["train"].select(range(5))))

print("\nTraining label counts:")
print(
    pd.Series(splits["train"]["label"])
    .value_counts(dropna=False)
    .sort_index()
)


Training data type: <class 'datasets.arrow_dataset.Dataset'>
Training column names: ['message_id', 'text', 'label', 'label_text', 'subject', 'message', 'date']

Sample training records:


,message_id,text,label,label_text,subject,message,date
0,4485,relocate to the usa ! get a green card easily ...,1,spam,relocate to the usa ! get a green card easily !,benthic immaturedirty eclogue conferringbract\...,2004-11-19
1,26745,new love tabs shop . visit our llcensed online...,1,spam,new love tabs shop .,visit our llcensed online dragstore for the be...,2005-07-07
2,3744,"miningnews . net newsletter - friday , january...",1,spam,"miningnews . net newsletter - friday , january...","friday , january 23 , 2004 miningnews . net\nt...",2004-01-23
3,10105,delivery notification : delivery has failed th...,1,spam,delivery notification : delivery has failed,this report relates to a message you sent with...,2005-06-24
4,7551,e - mail and voicemail retention policy as a r...,0,ham,e - mail and voicemail retention policy,"as a reminder , ena  , s policy regarding ret...",2000-11-07



Training label counts:
0    11681
1    10843
Name: count, dtype: int64


In [61]:
# Print dataset information
print('Vocabulary size:', len(vocabulary))
print('Maximum length:', MAX_LENGTH)
print('Dataset:', metadata['dataset_id'])
print('Dataset revision:', metadata['dataset_revision'])

Vocabulary size: 10000
Maximum length: 256
Dataset: ham_spam-20,000
Dataset revision: main


### 2.1. Encode Development Partitions

Only training and validation are encoded. Recreate the training loader for each seed; keep validation order fixed.

In [62]:
# Encode train and validation only
encoded_datasets = {
name: EncodedTextDataset(
        texts = splits[name]['text'],
        labels = splits[name]['label'],
        vocabulary = vocabulary,
        max_length = MAX_LENGTH) for name in ('train', 'validation')
}

# Keep validation order fixed
validation_loader = make_data_loader(encoded_datasets['validation'], batch_size = BATCH_SIZE, shuffle = False, seed = TRAINING_SEEDS[0])

# Inspect one validation batch
example_batch = next(iter(validation_loader))
for name, tensor in example_batch.items():
    print(name, tuple(tensor.shape))

# Recreate the training loader inside every seeded run, before training.
# Do not encode or evaluate the test partition during development.

tokens (64, 256)
lengths (64,)
labels (64,)


## Section 3: Baseline Model

## 3. Baseline Model

Implement a simple neural network baseline using the prepared email sequences and shared training vocabulary. Use the validation partition to select model settings and keep the test partition untouched during development.

In [63]:
# Section 3.1: Define the baseline model

PADDING_IDX = 0  # Must match the padding ID in shared preprocessing

class BaselineModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim=64, padding_idx=0):
        super().__init__()

        self.padding_idx = padding_idx

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=padding_idx
        )

        self.classifier = nn.Linear(embedding_dim, 2)

    def forward(self, input_ids):
        # Convert token IDs into embeddings
        embeddings = self.embedding(input_ids)

        # Ignore padding tokens when calculating the average
        mask = (input_ids != self.padding_idx).unsqueeze(-1)
        embeddings = embeddings * mask

        lengths = mask.sum(dim=1).clamp(min=1)
        pooled = embeddings.sum(dim=1) / lengths

        # Return scores for the two classes
        return self.classifier(pooled)


print("Baseline model class defined.")

Baseline model class defined.


## 4. Batch Inspection

Inspect one batch to verify the input keys, tensor shapes, and data types before defining the training loop.

In [68]:

# Section 4: Batch Inspection

print("Batch keys:", list(example_batch.keys()))

for key, value in example_batch.items():
    print(
        f"{key}: shape={tuple(value.shape)}, "
        f"dtype={value.dtype}, "
        f"sample={value.flatten()[:10].tolist()}"
    )

print("Vocabulary size:", len(vocabulary))
print("Maximum sequence length:", MAX_LENGTH)
print("Padding token ID:", vocabulary["<pad>"])
print("Unique labels:", torch.unique(example_batch["labels"]).tolist())

assert vocabulary["<pad>"] == 0
assert set(torch.unique(example_batch["labels"]).tolist()).issubset({0, 1})


Batch keys: ['tokens', 'lengths', 'labels']
tokens: shape=(64, 256), dtype=torch.int64, sample=[85, 8, 1, 1, 1174, 20, 1, 2, 917, 56]
lengths: shape=(64,), dtype=torch.int64, sample=[193, 256, 256, 256, 256, 256, 91, 162, 119, 256]
labels: shape=(64,), dtype=torch.int64, sample=[0, 0, 1, 1, 1, 0, 0, 0, 1, 0]
Vocabulary size: 10000
Maximum sequence length: 256
Padding token ID: 0
Unique labels: [0, 1]


## 5: Training and Validation

In [69]:

# Section 5: Training and Validation

import torch
import pandas as pd
from torch import nn

# 1. Training settings
EPOCHS = 10
LEARNING_RATE = 0.001
WEIGHT_DECAY = 0.01
GRADIENT_CLIP = 1.0

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_KEY = "tokens"
LABEL_KEY = "labels"
PADDING_IDX = 0

assert INPUT_KEY in example_batch
assert LABEL_KEY in example_batch
assert example_batch[INPUT_KEY].ndim == 2
assert example_batch[LABEL_KEY].ndim == 1
assert set(torch.unique(example_batch[LABEL_KEY]).tolist()).issubset({0, 1})

print("Device:", device)
print("Seeds:", TRAINING_SEEDS)
print("Batch size:", BATCH_SIZE)
print("Epochs:", EPOCHS)
print("Optimizer: AdamW")
print("Learning rate:", LEARNING_RATE)
print("Weight decay:", WEIGHT_DECAY)
print("Gradient clipping:", GRADIENT_CLIP)

training_history = []
seed_summaries = []

# 2. Train one model for each seed
for seed in TRAINING_SEEDS:

    print(f"\n{'=' * 50}")
    print(f"Training baseline with seed: {seed}")
    print(f"{'=' * 50}")

    set_seed(seed)

    train_loader = make_data_loader(
        encoded_datasets["train"],
        batch_size=BATCH_SIZE,
        shuffle=True,
        seed=seed
    )

    model = BaselineModel(
        vocab_size=len(vocabulary),
        embedding_dim=64,
        padding_idx=PADDING_IDX
    ).to(device)

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )

    best_f1 = -1.0
    best_epoch = 0
    best_checkpoint = CHECKPOINT_DIR / f"best_seed_{seed}.pt"

    # 3. Training epochs
    for epoch in range(1, EPOCHS + 1):

        model.train()
        total_train_loss = 0.0
        total_train_samples = 0

        for batch in train_loader:

            inputs = batch[INPUT_KEY].to(device).long()
            labels = batch[LABEL_KEY].to(device).long()

            optimizer.zero_grad()

            logits = model(inputs)
            loss = criterion(logits, labels)

            loss.backward()

            # Limit the gradient norm to help stabilize training
            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=GRADIENT_CLIP
            )

            optimizer.step()

            batch_size_actual = labels.size(0)
            total_train_loss += loss.item() * batch_size_actual
            total_train_samples += batch_size_actual

        train_loss = total_train_loss / max(total_train_samples, 1)

        # 4. Validation
        model.eval()

        total_val_loss = 0.0
        total_val_samples = 0
        correct = 0
        true_positive = 0
        false_positive = 0
        false_negative = 0

        with torch.no_grad():
            for batch in validation_loader:

                inputs = batch[INPUT_KEY].to(device).long()
                labels = batch[LABEL_KEY].to(device).long()

                logits = model(inputs)
                loss = criterion(logits, labels)
                predictions = logits.argmax(dim=1)

                n = labels.size(0)
                total_val_loss += loss.item() * n
                total_val_samples += n
                correct += (predictions == labels).sum().item()

                true_positive += (
                    (predictions == 1) & (labels == 1)
                ).sum().item()

                false_positive += (
                    (predictions == 1) & (labels == 0)
                ).sum().item()

                false_negative += (
                    (predictions == 0) & (labels == 1)
                ).sum().item()

        val_loss = total_val_loss / max(total_val_samples, 1)
        accuracy = correct / max(total_val_samples, 1)

        precision = true_positive / max(
            true_positive + false_positive, 1
        )
        recall = true_positive / max(
            true_positive + false_negative, 1
        )

        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall > 0 else 0.0
        )

        print(
            f"Seed {seed} | Epoch {epoch}/{EPOCHS} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Accuracy: {accuracy:.4f} | "
            f"Val Precision: {precision:.4f} | "
            f"Val Recall: {recall:.4f} | "
            f"Val F1: {f1:.4f}"
        )

        training_history.append({
            "seed": seed,
            "epoch": epoch,
            "train_loss": train_loss,
            "validation_loss": val_loss,
            "validation_accuracy": accuracy,
            "validation_precision": precision,
            "validation_recall": recall,
            "validation_f1": f1
        })

        # Keep the checkpoint with the best validation F1
        if f1 > best_f1:
            best_f1 = f1
            best_epoch = epoch

            torch.save({
                "seed": seed,
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "vocab_size": len(vocabulary),
                "embedding_dim": 64,
                "padding_idx": PADDING_IDX,
                "max_length": MAX_LENGTH,
                "validation_f1": best_f1,
                "optimizer": "AdamW",
                "learning_rate": LEARNING_RATE,
                "weight_decay": WEIGHT_DECAY,
                "gradient_clip": GRADIENT_CLIP,
                "epochs": EPOCHS,
                "batch_size": BATCH_SIZE
            }, best_checkpoint)

    seed_summaries.append({
        "seed": seed,
        "best_epoch": best_epoch,
        "best_validation_f1": best_f1,
        "checkpoint": str(best_checkpoint)
    })

# 5. Save results
history_df = pd.DataFrame(training_history)
summary_df = pd.DataFrame(seed_summaries)

history_path = RESULTS_DIR / "training_history.csv"
summary_path = RESULTS_DIR / "validation_summary.csv"

history_df.to_csv(history_path, index=False)
summary_df.to_csv(summary_path, index=False)

# Mean and sample standard deviation across the three seeds
mean_f1 = summary_df["best_validation_f1"].mean()
std_f1 = summary_df["best_validation_f1"].std(ddof=1)

print("\nTraining completed.")
print("\nBest validation results:")
display(summary_df)

print(f"Mean best validation F1: {mean_f1:.6f}")
print(f"Sample standard deviation: {std_f1:.6f}")

print("\nSaved files:")
print(history_path)
print(summary_path)
print("Checkpoints:", CHECKPOINT_DIR)


Device: cpu
Seeds: (42, 123, 2026)
Batch size: 64
Epochs: 10
Optimizer: AdamW
Learning rate: 0.001
Weight decay: 0.01
Gradient clipping: 1.0

Training baseline with seed: 42
Seed 42 | Epoch 1/10 | Train Loss: 0.4227 | Val Loss: 0.2262 | Val Accuracy: 0.9469 | Val Precision: 0.9268 | Val Recall: 0.9661 | Val F1: 0.9460
Seed 42 | Epoch 2/10 | Train Loss: 0.1539 | Val Loss: 0.1141 | Val Accuracy: 0.9712 | Val Precision: 0.9570 | Val Recall: 0.9845 | Val F1: 0.9705
Seed 42 | Epoch 3/10 | Train Loss: 0.0867 | Val Loss: 0.0771 | Val Accuracy: 0.9821 | Val Precision: 0.9732 | Val Recall: 0.9900 | Val F1: 0.9815
Seed 42 | Epoch 4/10 | Train Loss: 0.0593 | Val Loss: 0.0609 | Val Accuracy: 0.9844 | Val Precision: 0.9806 | Val Recall: 0.9871 | Val F1: 0.9838
Seed 42 | Epoch 5/10 | Train Loss: 0.0446 | Val Loss: 0.0521 | Val Accuracy: 0.9867 | Val Precision: 0.9842 | Val Recall: 0.9882 | Val F1: 0.9862
Seed 42 | Epoch 6/10 | Train Loss: 0.0352 | Val Loss: 0.0467 | Val Accuracy: 0.9885 | Val Precis

,seed,best_epoch,best_validation_f1,checkpoint
0,42,10,0.988772,C:\Users\Nouroz\Desktop\AI Machine Learning Se...
1,123,9,0.988227,C:\Users\Nouroz\Desktop\AI Machine Learning Se...
2,2026,10,0.988983,C:\Users\Nouroz\Desktop\AI Machine Learning Se...


Mean best validation F1: 0.988661
Sample standard deviation: 0.000390

Saved files:
C:\Users\Nouroz\Desktop\AI Machine Learning Semester Two\Advanced Deep Learning - AIGC-5500-0NA\Midtrem\AIGC5500-Midterm-Project\results\enron\baseline\training_history.csv
C:\Users\Nouroz\Desktop\AI Machine Learning Semester Two\Advanced Deep Learning - AIGC-5500-0NA\Midtrem\AIGC5500-Midterm-Project\results\enron\baseline\validation_summary.csv
Checkpoints: C:\Users\Nouroz\Desktop\AI Machine Learning Semester Two\Advanced Deep Learning - AIGC-5500-0NA\Midtrem\AIGC5500-Midterm-Project\checkpoints\enron\baseline
